# Cross-Dataset LOFO Confusion Matrices -- Baseline vs PC-Recenter

Confusion-matrix-only companion to `final_per_label_saliency.ipynb` (which bundles
this together with XAI/saliency plots for one fixed chip) -- this notebook is scoped
purely to confusion matrices, generalized across the whole cross-dataset LOFO
pipeline (`04_cross_dataset_training.py`/`08_cross_dataset_predict_new_chip.py`)
instead of one hardcoded dataset.

For every (group, curve_type, curve_alignment[, pc_ttp_anchor], train_center_frac,
outlier_filter, model) combination that's actually been trained: pools predictions
across every LOFO fold that combination has a saved model for (each chip predicted
by the model that genuinely excluded it -- the cross-dataset/LOFO equivalent of the
old notebook's "full 10-fold CV" pooling), and produces **two** confusion matrices
per combination: the baseline prediction, and the same prediction with
`--pc_recenter` applied. Both are computed the same way
`20260814-pc_recenter_combo_sweep.ipynb` computes them (via `08`'s own
`predict_new_chip`) rather than read back from the training-time results joblib --
so the two plots are a genuine apples-to-apples A/B comparison through identical
code, differing only in the recenter step.

Every plot is saved to disk (`confusion_matrices/` under that combination's own
`out_dir`, alongside `model_performance_{curve_type}/`); nothing renders inline --
there can be a lot of combinations. Skips (not errors) any combination that isn't
trained yet, same philosophy as the sweep notebook. PC excluded from every matrix
(control, not a target).

In [ ]:
import os
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix as sk_cm

try:
    notebook_path = globals().get('__vsc_ipynb_file__')
    if notebook_path:
        notebook_dir = os.path.dirname(os.path.dirname(notebook_path))
        os.chdir(notebook_dir)
except Exception as e:
    print(f"Could not change directory: {e}")

print("Current Working Directory:", os.getcwd())
%load_ext autoreload
%autoreload 2
import config

# Import-only -- none of these .py files are modified.
cdt = importlib.import_module("04_cross_dataset_training")
p08 = importlib.import_module("08_cross_dataset_predict_new_chip")
vis07 = importlib.import_module("07_attribution_vis_all")
rio = importlib.import_module("cross_dataset_result_io")

import tensorflow as tf
# See 20260814-pc_recenter_combo_sweep.ipynb -- DANN/CORAL models were compiled
# without jit_compile=False, so XLA's cudnn autotuner can spuriously fail even on
# an idle GPU. Disable globally so predicting on already-saved models doesn't hit it.
tf.config.optimizer.set_jit(False)

import joblib

## 1. Configuration -- edit these lists to change what gets covered

In [2]:
GROUPS_TO_TRY = list(config.CROSS_DATASET_GROUPS.keys())
EXP_FOLDER = config.FINAL_EXP_FOLDER + "_nc_subtract"
MODE_STR = "lofo"

CURVE_TYPES_TO_TRY = ["ori_curve_norm", "ori_curve_wavelet_bior35_norm"]

MODELS_TO_TRY = [
    "cnn_gru_dual",                          "cnn_gru_dual_dann",                     "cnn_gru_dual_coral",
    "cnn_gru_dual_supcon3",                  "cnn_gru_dual_supcon3_dann",             "cnn_gru_dual_supcon3_coral",
    "cnn_gru_dual_attn_recon",               "cnn_gru_dual_attn_recon_dann",          "cnn_gru_dual_attn_recon_coral",
    "cnn_gru_dual_attn_recon_supcon3",       "cnn_gru_dual_attn_recon_supcon3_dann",  "cnn_gru_dual_attn_recon_supcon3_coral",
]

OUTLIER_FILTERS_TO_TRY = ["none", "lofo_ae", "noamp_remove"]

# (curve_alignment, pc_ttp_anchor) -- anchor is ignored when alignment is acquisition_start
ALIGNMENTS_TO_TRY = [
    ("acquisition_start", "min"),
    ("pc_ttp", "min"),
    ("pc_ttp", "percentile"),
]

# None = full training pool (no center-crop); 0.5 = --train_center_frac 0.5
TRAIN_CENTER_FRACS_TO_TRY = [None, 0.5]

CONFUSION_MATRIX_SUBDIR = "confusion_matrices"

folder_names_by_group = {g: config.CROSS_DATASET_GROUPS[g] for g in GROUPS_TO_TRY}
exp_paths_by_group = {g: [Path(EXP_FOLDER, name) for name in names] for g, names in folder_names_by_group.items()}

def short_name(folder):
    return folder.split('_U_', 1)[1]

print(f"Groups ({len(GROUPS_TO_TRY)}): {GROUPS_TO_TRY}")

Groups (11): ['init_oneplex_v6', 'init_oneplex_nc_subtract', 'final_chip_1_2', 'final_4_chip_clean_nn', 'final_4_chip_cleanv2_nn', 'final_4_chip_cov_hadv_iav', 'final_4_chip_clean_nn_hpc', 'final_4_chip_cov_iav', 'final_4_chip_cov_nc', 'final_4_chip_iav_nc', 'final_4_chip_cov_iav_nc']


## 2. Helpers

`out_dir_for`/`aligned_chip` are the same as `20260814-pc_recenter_combo_sweep.ipynb`
(alignment depends on `group_name` because `config.LOFO_EXCLUDE_WELL_MAPPING`
excludes different wells per group). `ground_truth` is unchanged too.
`plot_confusion_matrix` is adapted from `final_per_label_saliency.ipynb`'s function
of the same name -- same visual style (row-normalised recall heatmap, count+% per
cell) -- but takes already-pooled `y_true`/`y_pred` string arrays directly instead of
reading a stored results dict, since both matrices here come from live
re-prediction, not training-time results. `label_names` defaults to the union of
whatever's actually in `y_true`/`y_pred` if not given, so nothing pooled gets
silently dropped even if a fold's own class vocabulary was narrower.

In [ ]:
def out_dir_for(group_name, curve_type, curve_alignment, pc_ttp_anchor):
    out_dir = Path(EXP_FOLDER) / "cross_dataset_cv" / group_name
    if curve_alignment == "pc_ttp":
        out_dir = out_dir / "curve_alignment_pc_ttp" / f"anchor_{pc_ttp_anchor}"
    return out_dir


_ALIGN_CACHE = {}

def aligned_chip(chip_path, group_name, curve_type, curve_alignment, pc_ttp_anchor):
    # chip_path is always this fold's own held-out chip here -- its name doubles as the
    # held_out_chip key for the fold-scoped resampler/recipe (see config.cross_dataset_alignment_dir).
    key = (chip_path.name, group_name, curve_type, curve_alignment, pc_ttp_anchor)
    if key not in _ALIGN_CACHE:
        out_dir = out_dir_for(group_name, curve_type, curve_alignment, pc_ttp_anchor)
        _ALIGN_CACHE[key] = p08.align_new_chip(chip_path, out_dir, curve_type, curve_alignment, pc_ttp_anchor,
                                               group_name=group_name, held_out_chip=chip_path.name)
    return _ALIGN_CACHE[key]


def ground_truth(chip_name, Y_well_raw, class_names):
    mapping = config.LABEL_MAPPINGS[chip_name]
    y_true = np.array([mapping.get(w, w) for w in Y_well_raw])
    if class_names:
        cn = list(class_names)
        y_true = np.array([next((c for c in cn if y == c or y.startswith(c + '-') or c.startswith(y + '-')), y)
                           for y in y_true])
    return y_true


def plot_confusion_matrix(y_true, y_pred, title, save_path, label_names=None):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    if label_names is None:
        label_names = sorted(set(y_true) | set(y_pred))

    cm = sk_cm(y_true, y_pred, labels=label_names)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm.astype(float), row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)
    n_cls = len(label_names)
    acc = (y_pred == y_true).mean() if len(y_true) else float('nan')

    fig, ax = plt.subplots(figsize=(9, 8), facecolor='white')
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Recall (row-normalised)', rotation=270, labelpad=14, fontsize=9)

    for i in range(n_cls):
        for j in range(n_cls):
            v = cm_norm[i, j]
            ax.text(j, i, f'{cm[i, j]}\n({v*100:.1f}%)',
                    ha='center', va='center', fontsize=12,
                    color='white' if v > 0.55 else '#1a1a2e')

    ax.set_xticks(range(n_cls))
    ax.set_yticks(range(n_cls))
    ax.set_xticklabels(label_names, rotation=40, ha='right', fontsize=9)
    ax.set_yticklabels(label_names, fontsize=9)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(f'{title}\naccuracy: {acc:.3f}  |  N={len(y_true)}', fontsize=11, fontweight='bold', pad=12)

    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)


print("Helpers defined.")

## 3. Sweep + save

For each (group, curve_type, alignment, frac, filter, model): pool baseline and
pc_recenter predictions across every held-out chip that combination has a model for,
then save both confusion matrices. Skips (silently, given the size of this
combination space) whenever a `.keras` file, results entry, or PC snapshot is
missing -- `class_names is None` specifically catches a fold whose only "hit" in the
results file was an always-merged `full_data` entry (see the sweep notebook for why
that happens with `train_center_frac`), not a real per-fold result.

In [ ]:
n_saved = 0
n_combos_with_data = 0

for group_name in GROUPS_TO_TRY:
    exp_paths_all = exp_paths_by_group[group_name]

    for curve_type in CURVE_TYPES_TO_TRY:
        for curve_alignment, pc_ttp_anchor in ALIGNMENTS_TO_TRY:
            out_dir = out_dir_for(group_name, curve_type, curve_alignment, pc_ttp_anchor)
            save_dir = out_dir / CONFUSION_MATRIX_SUBDIR

            for train_center_frac in TRAIN_CENTER_FRACS_TO_TRY:
                lofo_results = p08.load_partitioned(out_dir, MODE_STR, curve_type, train_center_frac=train_center_frac)
                if not any(k != "full_data" for k in lofo_results):
                    continue

                for filter_key_raw in OUTLIER_FILTERS_TO_TRY:
                    filter_key = "None" if filter_key_raw.lower() == "none" else filter_key_raw
                    filter_name_key = None if filter_key_raw.lower() == "none" else filter_key_raw

                    for model_key in MODELS_TO_TRY:
                        preds_key = config.MODEL_KEY_MAP.get(model_key, (None,))[0]
                        y_true_base, y_pred_base = [], []
                        y_true_rec, y_pred_rec = [], []

                        for chip_path in exp_paths_all:
                            chip_name = chip_path.name
                            fold_label = f"lofo_{chip_name}"
                            model_dir = out_dir / "model_interpretation" / fold_label
                            model_path = model_dir / f"{model_key}_{filter_key}_{curve_type}{rio.frac_suffix(train_center_frac)}_model.keras"
                            if not model_path.exists():
                                continue

                            filter_res_entry = lofo_results.get(fold_label, {}).get(filter_name_key, {})
                            if preds_key is None or preds_key not in filter_res_entry:
                                continue
                            class_names = lofo_results.get(fold_label, {}).get("class_names")
                            if class_names is None:
                                continue

                            align_result = aligned_chip(chip_path, group_name, curve_type, curve_alignment, pc_ttp_anchor)
                            if align_result is None:
                                continue
                            curves, resampler, Y_well_raw, pc_curves_aligned, coords, well_ids = align_result

                            loaded = vis07.load_saved_models(
                                model_dir, filter_key, len(resampler.t_grid), curve_type=curve_type, model_names=[model_key],
                                train_center_frac=train_center_frac)
                            if model_key not in loaded:
                                continue
                            model = loaded[model_key]

                            if p08._is_spatial(model_key) and (coords is None or well_ids is None):
                                continue

                            y_true = ground_truth(chip_name, Y_well_raw, class_names)
                            valid = y_true != "PC"
                            if not valid.any():
                                del model, loaded
                                tf.keras.backend.clear_session()
                                continue

                            exp_paths_train = [p for p in exp_paths_all if p.name != chip_name]
                            class_names_arr = np.array(class_names)

                            probs_base, _ = p08.predict_new_chip(
                                model, model_key, curves, coords, well_ids, pc_curves_aligned,
                                exp_paths_train, out_dir, curve_type, filter_key, curve_alignment,
                                pc_recenter=False, held_out_chip=chip_name)
                            pred_base = class_names_arr[np.argmax(probs_base, axis=1)]
                            y_true_base.append(y_true[valid]); y_pred_base.append(pred_base[valid])

                            try:
                                probs_r, _ = p08.predict_new_chip(
                                    model, model_key, curves, coords, well_ids, pc_curves_aligned,
                                    exp_paths_train, out_dir, curve_type, filter_key, curve_alignment,
                                    pc_recenter=True, force_rerun=True, held_out_chip=chip_name)
                                pred_r = class_names_arr[np.argmax(probs_r, axis=1)]
                                y_true_rec.append(y_true[valid]); y_pred_rec.append(pred_r[valid])
                            except ValueError:
                                pass  # no PC snapshot for this chip -- just skip it for the recenter pool

                            del model, loaded
                            tf.keras.backend.clear_session()

                        if not y_true_base:
                            continue
                        n_combos_with_data += 1

                        frac_tag = f"_center{train_center_frac:g}" if train_center_frac is not None else ""
                        anchor_tag = f"_{pc_ttp_anchor}" if curve_alignment == "pc_ttp" else ""
                        tag = f"{model_key}_{filter_key}_{curve_type}_{curve_alignment}{anchor_tag}{frac_tag}"
                        meta = f"filter={filter_key_raw} | {curve_alignment}{anchor_tag} | frac={train_center_frac}"

                        y_true_base_all = np.concatenate(y_true_base)
                        y_pred_base_all = np.concatenate(y_pred_base)
                        plot_confusion_matrix(
                            y_true_base_all, y_pred_base_all,
                            f"{group_name} | {model_key} | {meta} | BASELINE",
                            save_dir / f"{tag}_confusion_baseline.png")
                        n_saved += 1
                        print(f"  [saved] {group_name} / {tag}_confusion_baseline.png  (N={len(y_true_base_all)})")

                        if y_true_rec:
                            y_true_rec_all = np.concatenate(y_true_rec)
                            y_pred_rec_all = np.concatenate(y_pred_rec)
                            plot_confusion_matrix(
                                y_true_rec_all, y_pred_rec_all,
                                f"{group_name} | {model_key} | {meta} | PC-RECENTER",
                                save_dir / f"{tag}_confusion_pc_recenter.png")
                            n_saved += 1
                            print(f"  [saved] {group_name} / {tag}_confusion_pc_recenter.png  (N={len(y_true_rec_all)})")

print(f"\n{n_saved} confusion matrix PNGs saved across {n_combos_with_data} (group, curve_type, alignment, frac, filter, model) combinations.")